# Example 10.4.3 / 6: Validate Against the Original Function

Companion notebook for **10.4.3 Automatic Piecewise-Linear Approximation** in the AMPL lab Documentation, section "Validate Against the Original Function". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Validate Against the Original Function

Between breakpoints the PWL graph differs from the true function, so a point that satisfies the
approximation can violate the original constraint. Two plants each produce `exp(Level[p])` units,
the total must reach 50, and total effort is minimized. The constraint has a convex function on the
large side of `>=`, so the feasible region is nonconvex and the driver needs discrete structure:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;  # plants producing output
param demand > 0;  # total output required
var Level {PLANTS} >= 0, <= 10;  # effort at each plant

minimize Effort: sum {p in PLANTS} Level[p];  # total effort
# nonconvex: convex function on the >= side
subject to Output: sum {p in PLANTS} exp(Level[p]) >= demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["north", "south"]
ampl.param["demand"] = 50

In [5]:
# run.py
ampl.solve(solver="highs", highs_options="cvt:plapprox:reltol=0.1")
print(ampl.solve_result)
print(ampl.var["Level"].to_dict())
print(ampl.con["Output"].body())

HiGHS 1.14.0:   cvt:plapprox:reltol = 0.10000000000000001

------------ WARNINGS ------------
WARNING.  2 case(s) of "PLApprox_Exp". One of them:
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.100000).
HiGHS 1.14.0: optimal solution; objective 3.825275216
296 simplex iterations
1 branching nodes
 
------------ WARNINGS ------------
WARNING.  2 case(s) of "PLApprox_Exp". One of them:
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.100000).
  Type                         MaxAbs [Name]   MaxRel [Name]
* expr '_exp'                  3E+00           6E-02         
*: Using the solver's aux variable values.
Documentation: mp.ampl.com/modeling-tools.html#automatic-solution-check.
solved
{'north': 0, 'south': 3.825275216223519}
46.84541603250196


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PLANTS := north south;
param demand := 50;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option highs_options 'cvt:plapprox:reltol=0.1';
solve;
display Level, Output.body;

HiGHS 1.14.0:   cvt:plapprox:reltol = 0.10000000000000001

------------ WARNINGS ------------
WARNING.  2 case(s) of "PLApprox_Exp". One of them:
  An expression of type 'Exp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.100000).
Level [*] :=
north  0
south  3.82528
;

Output.body = 46.8454



### Another way in amplpy

The status is `solved`, yet `Output.body`, which AMPL recomputes from the true `exp`, is below 50.
In the amplpy run's full log, MP's automatic solution check reports the same thing as a
"Tolerance violations" warning on `expr '_exp'`, with the maximum absolute and relative
violation. (The full log repeats the approximation warning after HiGHS's result line; the
pure-AMPL run sets `solver_msg 0` and shows it once.) (The exact optimum puts
$\log 49 \approx 3.892$ on one plant and 0 on the other.)

To turn such a violation into a status instead of a warning, add `sol:chk:fail`, which makes a
failed check return solve result 150 (`solved?`). Tightening the tolerance shrinks the violation.
With the same model and data:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PLANTS"] = ["north", "south"]
ampl.param["demand"] = 50

In [11]:
# run.py
ampl.solve(solver="highs", verbose=False,
           highs_options="cvt:plapprox:reltol=0.1 sol:chk:fail")
print(ampl.solve_result, ampl.solve_result_num)

ampl.solve(solver="highs", verbose=False,
           highs_options="cvt:plapprox:reltol=0.0001")
print(ampl.solve_result, ampl.con["Output"].body())

solved? 150
solved 49.99615986840084


### Pure AMPL: data.dat and model.run

Pure AMPL, with the same `model.mod`:

In [12]:
%%writefile data_2.dat
set PLANTS := north south;
param demand := 50;

Writing data_2.dat


In [13]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data_2.dat;

In [14]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
option highs_options 'cvt:plapprox:reltol=0.1 sol:chk:fail';
solve > highs.log;
display solve_result, solve_result_num;
option highs_options 'cvt:plapprox:reltol=0.0001';
solve > highs.log;
display solve_result, Output.body;

solve_result = 'solved?'
solve_result_num = 150

solve_result = solved
Output.body = 49.9962

